# Station Stacking v11 Settlement Fix - KATL

Experimental notebook for `KATL`.

This experiment keeps the settlement-first v11 remaining-warmup contract, removes features above 3% missingness within each training fold, adds the expanded live-safe 11 AM forecast-temperature feature family, and writes artifacts to `data/calibration/station_stacking_v11_settlement_fix`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KATL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v11_settlement_fix_temp_stack"
EXPORT_MODEL_WEIGHTS = False
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS,
    _fit_feature_columns,
    _modeling_frame,
    V11_DROPPED_FEATURE_COLUMNS,
    V11_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V11 Contract

`feature_version="v11_settlement_fix_temp"` keeps the v9 feature contract and remaining-warmup target, but trains base learners with Huber-style objectives while retaining the ridge stack selected by validation MAE.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V11_FEATURE_COLUMNS, sorted(V11_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
0,KATL,gfs,1999,2021-01-01,2026-07-12
1,KATL,hrrr,1998,2021-01-01,2026-06-21
2,KATL,nbm,1998,2021-01-01,2026-07-12


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v11_settlement_fix_temp",
    target_mode="remaining_warmup",
    target_source="settlement_first",
    max_feature_missing_fraction=0.03,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11_settlement_fix",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v11_settlement_fix/KATL_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


D:\dev\weather-research\src\calibration\station_stacking.py:3134: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:3133: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:3134: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,731,1.304192,1.835054
1,validation_2024_2025,lightgbm,731,1.293397,1.775257
2,validation_2024_2025,catboost,731,1.318954,1.787858
3,validation_2024_2025,provider_mean,731,2.127199,3.107267
4,validation_2024_2025,provider_median,731,2.044175,3.004660
5,validation_2024_2025,nbm_raw,731,2.006541,2.889070
6,validation_2024_2025,hrrr_raw,731,2.817351,4.029933
7,validation_2024_2025,gfs_raw,731,2.883423,3.907159
8,test_2026,xgboost,172,1.320866,1.874673
9,test_2026,lightgbm,172,1.353778,1.878237


In [8]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline="notebooks/station_stacking_v11_settlement_fix",
    )

    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this experimental notebook.")


Model export disabled for this experimental notebook.


## V11 Feature Coverage


In [9]:
v11_feature_coverage = (
    result.features[V11_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v11_feature_coverage


,feature,coverage_pct
0,v2_spread_per_warmup_f,100.000000
1,v2_morning_warmup_to_consensus_f,100.000000
2,v3_remaining_warmup_from_high_so_far_f,100.000000
3,v3_high_so_far_above_current_f,100.000000
4,v2_humidity_warmup_interaction,100.000000
5,v4_observed_precip_recent_mm_est,100.000000
6,v4_forecast_wet_observed_dry,100.000000
7,v4_forecast_observed_precip_match,100.000000
8,v3_humidity_remaining_warmup_interaction,100.000000
9,v3_remaining_warmup_per_spread_f,100.000000


In [10]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V11_FEATURE_COLUMNS)]


,feature,kind
20,observed_temp_change_last_1h_f,numeric
21,observed_temp_change_last_3h_f,numeric
22,observed_morning_warmup_rate_f_per_hour,numeric
23,observed_high_so_far_change_since_9am_f,numeric
163,v2_recent_heat_anomaly_f,numeric
164,v2_recent_heat_momentum_f,numeric
165,v2_morning_warmup_to_consensus_f,numeric
166,v2_consensus_minus_7d_actual_f,numeric
167,v2_spread_per_warmup_f,numeric
168,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [11]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V11_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [12]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,99.8999
1,observed_temp_change_last_3h_f,99.8999
2,observed_morning_warmup_rate_f_per_hour,99.8999
3,observed_high_so_far_change_since_9am_f,99.8999


## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
0,oof_2026,catboost,172,126,73.255814
7,oof_2026,ridge_stack,172,119,69.186047
3,oof_2026,lightgbm,172,118,68.604651
8,oof_2026,xgboost,172,115,66.860465
4,oof_2026,nbm_raw,172,92,53.488372
6,oof_2026,provider_median,172,85,49.418605
5,oof_2026,provider_mean,172,79,45.930233
2,oof_2026,hrrr_raw,172,72,41.860465
1,oof_2026,gfs_raw,172,59,34.302326
12,validation_2024_2025,lightgbm,731,497,67.989056


## Version Comparison


In [14]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,lightgbm,137,1.529121,2.224566,v11
1,test_2026,ridge_stack,137,1.531900,2.209782,v11
2,test_2026,lightgbm,125,1.547989,2.093915,v5
3,test_2026,ridge_stack,125,1.566708,2.118374,v5
4,test_2026,xgboost,125,1.569277,2.117086,v5
...,...,...,...,...,...,...
111,validation_2024_2025,gfs_raw,541,3.557627,5.060559,v1
112,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v2
113,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v3
114,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v4


## 2026 OOF Weather Brackets


In [15]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bucket_log_loss,bracket_accuracy_pct,p95_absolute_error_f,large_miss_5f_pct
0,xgboost,172,1.320866,1.874673,1.411028,42.44186,3.415366,1.744186
1,lightgbm,172,1.353778,1.878237,1.409215,43.604651,3.558558,1.744186
2,catboost,172,1.346584,1.828936,1.385114,43.604651,3.856347,1.744186
3,ridge_stack,172,1.319281,1.816237,1.378067,42.44186,3.743977,1.744186
4,provider_mean,172,2.139412,3.328671,1.874295,34.883721,5.354214,6.395349
5,provider_median,172,2.119176,3.267309,1.872088,34.302326,5.901595,6.976744
6,nbm_raw,172,2.082419,3.220885,1.869031,35.465116,5.901595,7.558140
7,hrrr_raw,172,2.529610,3.819664,1.973448,30.232558,7.030181,12.209302
8,gfs_raw,172,2.842734,3.969046,2.081050,21.511628,7.406898,12.790698


## Train-Fold 3% Missingness Audit


In [16]:
modeling_frame, candidate_categorical, candidate_numeric = _modeling_frame(result.features, config)
candidate_features = [*candidate_categorical, *candidate_numeric]
audit_specs = [
    (fold.name, fold.train_start_year, fold.train_end_year)
    for fold in YEAR_SPLIT_EXPANDING_FOLDS
] + [("test_refit_2021_2025", 2021, 2025)]

missingness_rows = []
years = pd.to_numeric(modeling_frame["year"], errors="coerce")
for fold_name, train_start, train_end in audit_specs:
    train = modeling_frame.loc[years.between(train_start, train_end)].copy()
    retained_categorical, retained_numeric = _fit_feature_columns(
        train,
        candidate_categorical,
        candidate_numeric,
        max_missing_fraction=config.effective_max_feature_missing_fraction,
    )
    retained = set(retained_categorical) | set(retained_numeric)
    for feature in candidate_features:
        numeric_feature = feature in candidate_numeric
        values = pd.to_numeric(train[feature], errors="coerce") if numeric_feature else train[feature]
        missingness_rows.append(
            {
                "fold": fold_name,
                "train_start_year": train_start,
                "train_end_year": train_end,
                "feature": feature,
                "kind": "numeric" if numeric_feature else "categorical",
                "missing_fraction": float(values.isna().mean()),
                "retained": feature in retained,
            }
        )

fold_feature_missingness = pd.DataFrame(missingness_rows)
retained_dropped_summary = (
    fold_feature_missingness.groupby(["fold", "retained"], as_index=False)
    .agg(feature_count=("feature", "nunique"), maximum_missing_fraction=("missing_fraction", "max"))
)
fold_feature_missingness.to_csv(config.resolved_output_dir() / f"{STATION_ID}_fold_feature_missingness.csv", index=False)
retained_dropped_summary, fold_feature_missingness.loc[~fold_feature_missingness["retained"]].sort_values(
    ["fold", "missing_fraction"], ascending=[True, False]
)


(                     fold  retained  feature_count  maximum_missing_fraction
 0  fold_2021_2023_to_2024     False             30                  0.961574
 1  fold_2021_2023_to_2024      True            198                  0.021958
 2  fold_2021_2024_to_2025     False             30                  0.954764
 3  fold_2021_2024_to_2025      True            198                  0.016450
 4    test_refit_2021_2025     False             30                  0.950110
 5    test_refit_2021_2025      True            198                  0.013158,
                        fold  train_start_year  train_end_year  \
 10   fold_2021_2023_to_2024              2021            2023   
 11   fold_2021_2023_to_2024              2021            2023   
 1    fold_2021_2023_to_2024              2021            2023   
 9    fold_2021_2023_to_2024              2021            2023   
 39   fold_2021_2023_to_2024              2021            2023   
 ..                      ...               ...           

## Expanded 11 AM Feature Coverage and Provider Count


In [17]:
new_feature_coverage = (
    result.features[V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)
provider_count_coverage = (
    result.features["v11sf_forecast_temp_11am_provider_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("available_provider_count")
    .reset_index(name="row_count")
)
provider_count_coverage["row_pct"] = provider_count_coverage["row_count"] / len(result.features) * 100
new_feature_coverage.to_csv(config.resolved_output_dir() / f"{STATION_ID}_11am_feature_coverage.csv", index=False)
new_feature_coverage, provider_count_coverage


(                                              feature  coverage_pct
 0                     v11sf_forecast_temp_11am_mean_f         100.0
 1                   v11sf_forecast_temp_11am_median_f         100.0
 2           v11sf_forecast_temp_11am_minus_observed_f         100.0
 3                v11sf_forecast_temp_11am_abs_error_f         100.0
 4               v11sf_forecast_temp_11am_warm_error_f         100.0
 5               v11sf_forecast_temp_11am_cool_error_f         100.0
 6                   v11sf_forecast_temp_11am_spread_f         100.0
 7             v11sf_forecast_temp_11am_provider_count         100.0
 8   v11sf_forecast_temp_bias_remaining_warmup_inte...         100.0
 9          v11sf_observation_adjusted_provider_high_f         100.0
 10                 v11sf_forecast_warmup_after_11am_f         100.0,
    available_provider_count  row_count    row_pct
 0                         1         70   3.503504
 1                         2        480  24.024024
 2                

## New-Feature Importance


In [18]:
new_feature_importance = result.feature_importance.loc[
    result.feature_importance["feature"].isin(V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS)
].sort_values(["method", "importance_mean_mae_f"], ascending=[True, False])
new_feature_importance


,method,param_key,feature,importance_mean_mae_f,importance_std_mae_f,n_repeats,train_start_year,train_end_year,test_year,train_rows,test_rows
11,catboost,trial_19,v11sf_forecast_warmup_after_11am_f,0.048262,0.020839,10,2021,2025,2026,1824,172
60,catboost,trial_19,v11sf_forecast_temp_bias_remaining_warmup_inte...,0.006700,0.003233,10,2021,2025,2026,1824,172
73,catboost,trial_19,v11sf_forecast_temp_11am_spread_f,0.005279,0.001993,10,2021,2025,2026,1824,172
88,catboost,trial_19,v11sf_observation_adjusted_provider_high_f,0.003984,0.002893,10,2021,2025,2026,1824,172
164,catboost,trial_19,v11sf_forecast_temp_11am_warm_error_f,0.001570,0.002518,10,2021,2025,2026,1824,172
213,catboost,trial_19,v11sf_forecast_temp_11am_minus_observed_f,0.000799,0.003214,10,2021,2025,2026,1824,172
235,catboost,trial_19,v11sf_forecast_temp_11am_median_f,0.000584,0.001988,10,2021,2025,2026,1824,172
261,catboost,trial_19,v11sf_forecast_temp_11am_mean_f,0.000332,0.001563,10,2021,2025,2026,1824,172
319,catboost,trial_19,v11sf_forecast_temp_11am_provider_count,0.000000,0.000000,10,2021,2025,2026,1824,172
453,catboost,trial_19,v11sf_forecast_temp_11am_abs_error_f,-0.000552,0.001867,10,2021,2025,2026,1824,172


## 2026 Monthly Metrics


In [19]:
monthly_predictions = result.test_predictions.copy()
monthly_predictions["month"] = pd.to_datetime(monthly_predictions["contract_date"], errors="coerce").dt.month
monthly_metrics = (
    monthly_predictions.dropna(subset=["month", "error_f"])
    .groupby(["method", "month"], as_index=False)
    .agg(
        count=("error_f", "size"),
        mae_f=("absolute_error_f", "mean"),
        rmse_f=("error_f", lambda values: float(np.sqrt(np.mean(np.square(values))))),
        bias_f=("error_f", "mean"),
    )
)
monthly_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_2026_monthly_metrics.csv", index=False)
monthly_metrics


,method,month,count,mae_f,rmse_f,bias_f
0,catboost,1,31,1.257833,1.551392,-0.224715
1,catboost,2,28,1.459864,1.865235,0.340127
2,catboost,3,31,1.167310,1.452479,0.184392
3,catboost,4,30,1.588198,2.153301,-0.549152
4,catboost,5,31,1.567231,2.326103,0.224659
5,catboost,6,21,0.920320,1.216557,-0.091297
6,gfs_raw,1,31,2.802349,3.395373,2.287120
7,gfs_raw,2,28,3.106535,3.827184,2.926251
8,gfs_raw,3,31,3.190084,5.440344,1.777883
9,gfs_raw,4,30,2.083964,2.684767,-0.458210


## Performance by Warm/Cool 11 AM Forecast Delta


In [20]:
delta_by_date = result.features[[
    "contract_date",
    "v11sf_forecast_temp_11am_minus_observed_f",
]].copy()
delta_predictions = result.test_predictions.merge(delta_by_date, on="contract_date", how="left")
delta_predictions["forecast_temp_delta_bucket"] = pd.cut(
    delta_predictions["v11sf_forecast_temp_11am_minus_observed_f"],
    bins=[-np.inf, -2.0, -0.5, 0.5, 2.0, np.inf],
    labels=["cool_gt_2f", "cool_0.5_to_2f", "near_match", "warm_0.5_to_2f", "warm_gt_2f"],
)
warm_cool_metrics = (
    delta_predictions.dropna(subset=["forecast_temp_delta_bucket", "error_f"])
    .groupby(["method", "forecast_temp_delta_bucket"], observed=True, as_index=False)
    .agg(count=("error_f", "size"), mae_f=("absolute_error_f", "mean"), bias_f=("error_f", "mean"))
)
warm_cool_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_warm_cool_delta_metrics.csv", index=False)
warm_cool_metrics


,method,forecast_temp_delta_bucket,count,mae_f,bias_f
0,catboost,cool_gt_2f,34,1.599619,0.087282
1,catboost,cool_0.5_to_2f,55,0.914258,0.068621
2,catboost,near_match,39,1.467832,-0.343615
3,catboost,warm_0.5_to_2f,29,1.418158,-0.073326
4,catboost,warm_gt_2f,15,1.904614,0.375462
5,gfs_raw,cool_gt_2f,34,3.787576,3.739109
6,gfs_raw,cool_0.5_to_2f,55,2.845005,1.701259
7,gfs_raw,near_match,39,2.583485,-0.293216
8,gfs_raw,warm_0.5_to_2f,29,2.390429,-0.388416
9,gfs_raw,warm_gt_2f,15,2.241267,-1.095862


## Common-Date Comparison with Existing V11 Settlement


In [21]:
baseline_path = (
    PROJECT_ROOT
    / "data"
    / "calibration"
    / "station_stacking_v11_settlement"
    / f"{STATION_ID}_year_split_test_predictions.csv"
)
baseline_predictions = pd.read_csv(baseline_path)
baseline_predictions["contract_date"] = baseline_predictions["contract_date"].astype(str).str[:10]
fix_predictions = result.test_predictions.copy()
fix_predictions["contract_date"] = fix_predictions["contract_date"].astype(str).str[:10]
comparison = baseline_predictions.merge(
    fix_predictions,
    on=["contract_date", "method"],
    suffixes=("_baseline", "_fix"),
)
comparison["baseline_abs_error_f"] = (
    pd.to_numeric(comparison["actual_high_f_baseline"], errors="coerce")
    - pd.to_numeric(comparison["predicted_high_f_baseline"], errors="coerce")
).abs()
comparison["fix_abs_error_f"] = (
    pd.to_numeric(comparison["actual_high_f_fix"], errors="coerce")
    - pd.to_numeric(comparison["predicted_high_f_fix"], errors="coerce")
).abs()
common_date_comparison = (
    comparison.groupby("method", as_index=False)
    .agg(
        common_date_count=("contract_date", "size"),
        baseline_mae_f=("baseline_abs_error_f", "mean"),
        fix_mae_f=("fix_abs_error_f", "mean"),
        fix_better_days=("fix_abs_error_f", lambda values: int((values < comparison.loc[values.index, "baseline_abs_error_f"]).sum())),
        baseline_better_days=("fix_abs_error_f", lambda values: int((values > comparison.loc[values.index, "baseline_abs_error_f"]).sum())),
    )
)
common_date_comparison["delta_mae_f"] = common_date_comparison["fix_mae_f"] - common_date_comparison["baseline_mae_f"]
common_date_comparison.to_csv(config.resolved_output_dir() / f"{STATION_ID}_v11_common_date_comparison.csv", index=False)
common_date_comparison.sort_values("delta_mae_f")


,method,common_date_count,baseline_mae_f,fix_mae_f,fix_better_days,baseline_better_days,delta_mae_f
1,gfs_raw,172,2.842734,2.842734,0,0,0.000000
2,hrrr_raw,172,2.529610,2.529610,0,0,0.000000
5,provider_mean,172,2.139412,2.139412,3,2,0.000000
4,nbm_raw,172,2.082419,2.082419,0,0,0.000000
6,provider_median,172,2.119176,2.119176,0,0,0.000000
7,ridge_stack,172,1.294074,1.319281,89,83,0.025207
8,xgboost,172,1.276827,1.320866,81,89,0.044039
0,catboost,172,1.296958,1.346584,77,93,0.049626
3,lightgbm,172,1.297384,1.353778,68,99,0.056393
